# 1. Project Introduction

Welcome! In this notebook, we will explore **AdaBoost** (Adaptive Boosting), one of the earliest and most popular boosting algorithms.

### What is AdaBoost?
* It is a **supervised learning** classifier.
* Boosting is a sequential ensemble method. Rather than training trees independently (like Random Forest), AdaBoost trains them **one after another**.
* Each successive tree (often a very simple tree with a depth of 1, called a **decision stump**) focuses on correcting the errors made by the previous trees.
* **Adaptive**: It does this by increasing the weights of misclassified data points, so the next stump pays more attention to hard cases.

### Why does it exist?
* It turns weak learners (models that perform just slightly better than random guessing) into a strong collective ensemble.

### Real-World Use Cases:
* **Facial Detection**: Historically used in early face-detection software (e.g., Viola-Jones algorithm).
* **Customer Churn**: Predicting subscriber drop-off.


# 2. Problem Statement

* **Goal**: Predict if a credit card user will **Default (1)** on their next payment or pay **On Time (0)**.
* **Business Value**: Minimizes banking losses from credit defaults.


In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn import metrics


# 4. Create Synthetic Dataset

We define features for **100 credit card accounts**.
* **Age**: Account holder's age.
* **Monthly_Income_K**: Monthly income in k$.
* **Missed_Payments_Count**: Number of billing cycles missed in the past year (0 to 6).
* **Will_Default**: Target classification label.


In [ ]:
# Hardcoded customer default dataset
age = [
    25, 45, 30, 55, 22, 40, 60, 27, 33, 48, 26, 35, 52, 29, 42, 50, 31, 38, 44, 23,
    26, 46, 31, 56, 23, 41, 61, 28, 34, 49, 27, 36, 53, 30, 43, 51, 32, 39, 45, 24,
    24, 44, 29, 54, 21, 39, 59, 26, 32, 47, 25, 34, 51, 28, 41, 49, 30, 37, 43, 22,
    25, 45, 30, 55, 22, 40, 60, 27, 33, 48, 26, 35, 52, 29, 42, 50, 31, 38, 44, 23,
    30, 35, 40, 45, 50, 25, 28, 32, 38, 42, 48, 52, 55, 60, 22, 24, 29, 31, 34, 37
]

income = [
    3.0, 8.5, 4.2, 12.0, 2.5, 6.0, 15.0, 3.8, 5.5, 9.0, 4.0, 7.5, 11.0, 5.0, 8.0, 10.0, 6.5, 7.0, 9.5, 3.2,
    3.2, 8.7, 4.4, 12.2, 2.7, 6.2, 15.2, 4.0, 5.7, 9.2, 4.2, 7.7, 11.2, 5.2, 8.2, 10.2, 6.7, 7.2, 9.7, 3.4,
    2.8, 8.3, 4.0, 11.8, 2.3, 5.8, 14.8, 3.6, 5.3, 8.8, 3.8, 7.3, 10.8, 4.8, 7.8,  9.8, 6.3, 6.8, 9.3, 3.0,
    3.0, 8.5, 4.2, 12.0, 2.5, 6.0, 15.0, 3.8, 5.5, 9.0, 4.0, 7.5, 11.0, 5.0, 8.0, 10.0, 6.5, 7.0, 9.5, 3.2,
    4.0, 5.0, 6.0, 7.0,  8.0, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5,  9.5, 10.5, 12.0, 2.8, 3.0, 4.2, 4.8, 5.2, 5.8
]

missed_payments = [
    2, 0, 1, 0, 4, 1, 0, 3, 2, 0, 3, 1, 0, 2, 1, 0, 1, 1, 0, 3,
    2, 0, 1, 0, 4, 1, 0, 3, 2, 0, 3, 1, 0, 2, 1, 0, 1, 1, 0, 3,
    2, 0, 1, 0, 4, 1, 0, 3, 2, 0, 3, 1, 0, 2, 1, 0, 1, 1, 0, 3,
    2, 0, 1, 0, 4, 1, 0, 3, 2, 0, 3, 1, 0, 2, 1, 0, 1, 1, 0, 3,
    1, 2, 0, 1, 0, 3, 2, 1, 0, 1, 2, 0, 1, 0, 4, 3, 2, 1, 0, 1
]

will_default = [
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1,
    0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0
]

df = pd.DataFrame({
    'Age': age,
    'Monthly_Income_K': income,
    'Missed_Payments_Count': missed_payments,
    'Will_Default': will_default
})

print("Shape:", df.shape)
print(df.head())


# 5. Exploratory Data Analysis (EDA)


In [ ]:
# Chart 1: Distribution of Missed Payments by Default Outcome
plt.figure(figsize=(8, 4))
sns.boxplot(x='Will_Default', y='Missed_Payments_Count', data=df, palette='Set2')
plt.title('Missed Payments vs. Default Outcome')
plt.xlabel('Will Default (0 = No, 1 = Yes)')
plt.ylabel('Missed Payments Count')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* Defaulters (class 1) have a median of 2-3 missed payments, while non-defaulters have 0 or 1.


In [ ]:
# Cleaning check
print("Null count:", df.isnull().sum().sum())


In [ ]:
# Feature Selection
X = df[['Age', 'Monthly_Income_K', 'Missed_Payments_Count']]
y = df['Will_Default']


In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 9. Model Building

* **How it works**: AdaBoost trains a sequence of decision stumps (one-split trees). After each stump is built, the weights of incorrectly classified training records are increased. The final prediction is a weighted sum of predictions from all stumps.


In [ ]:
# Initialize AdaBoost classifier with 30 stumps
model = AdaBoostClassifier(n_estimators=30, random_state=42)


In [ ]:
# Train AdaBoost
model.fit(X_train, y_train)


In [ ]:
# Predict labels
predictions = model.predict(X_test)


In [ ]:
# Compute metrics
accuracy = metrics.accuracy_score(y_test, predictions)
precision = metrics.precision_score(y_test, predictions)
recall = metrics.recall_score(y_test, predictions)
f1 = metrics.f1_score(y_test, predictions)
conf_matrix = metrics.confusion_matrix(y_test, predictions)

# Print metrics in plain English
print(f"Accuracy Score: {accuracy:.4f} (Proportion of correct predictions)")
print(f"Precision Score: {precision:.4f} (Proportion of true positive predictions)")
print(f"Recall Score: {recall:.4f} (Proportion of actual positives caught)")
print(f"F1 Score: {f1:.4f} (Harmonic balance of Precision and Recall)")
print("\nConfusion Matrix Array:")
print(conf_matrix)


# 13. Visualizing Model Performance

We will plot:
1. **Confusion Matrix Heatmap**.
2. **Feature Importance Plot**: Showcasing feature contribution across stumps.


In [ ]:
# Plot 1: Confusion Matrix Heatmap
conf_matrix = metrics.confusion_matrix(y_test, predictions)
plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Oranges', 
            xticklabels=['Predicted Active', 'Predicted Default'], 
            yticklabels=['Actual Active', 'Actual Default'])
plt.title('AdaBoost Confusion Matrix')
plt.show()


In [ ]:
# Plot 2: Feature Importances
plt.figure(figsize=(6, 4))
sns.barplot(x=model.feature_importances_, y=X.columns, palette='copper')
plt.title('AdaBoost Feature Importances')
plt.xlabel('Importance score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()


### What Did We Observe?
* `Missed_Payments_Count` is identified as the most important feature.


# 14. Model Interpretation

* **Sequential Learning**: AdaBoost builds successive stumps where each stump's voting power depends on its error rate.
* **Feature Importance**: Stumps split on features that clear up prediction errors for weighted records.


# 15. Conclusion
* AdaBoost builds predictive capability iteratively.


# 16. Beginner ML Dictionary

Here are simple, one-sentence explanations of common Machine Learning terms to help you review:

* **Feature**: An input variable or column in your dataset used to make predictions (e.g., hours studied).
* **Target**: The output variable or label you want the model to predict (e.g., final exam score).
* **Training Data**: The portion of the dataset used to teach the model and find patterns.
* **Testing Data**: The portion of the dataset held back to evaluate how well the model performs on new, unseen data.
* **Prediction**: The output value generated by the trained model when given new input features.
* **Overfitting**: A scenario where the model learns the training data too well, including its noise, and performs poorly on new data.
* **Underfitting**: A scenario where the model is too simple to learn the underlying patterns in the training data, leading to poor performance on both training and test data.
* **Model**: The mathematical representation of the patterns learned from the training data by the algorithm.
* **Algorithm**: The set of rules or mathematical procedures followed to build the model from the data (e.g., Linear Regression).
* **Accuracy**: The percentage of correct predictions made by a classification model.
* **Cluster**: A group of similar data points grouped together by an unsupervised learning algorithm based on their characteristics.
* **Centroid**: The center point of a cluster, representing the average location of all data points belonging to that cluster.
